<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/hybrid_reference_flow_and_density_ratios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Native reference flow and density ratios

This notebook is the YAML-driven replacement for the original Exercise 5 workflow. It uses only `hnsbi-toolkit`: the ratio trainer, diagnostics, ONNX export, workspace writer, JAX likelihood, and Minuit inference are all native.

> On a fresh Colab runtime, the setup cell may restart the kernel once to align NumPy with the LHC environment. After Colab reconnects, run the setup cell again.

In [1]:
from importlib.metadata import version as installed_version
from pathlib import Path
import os, signal, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    loaded_numpy = sys.modules.get('numpy')
    loaded_numpy_version = getattr(loaded_numpy, '__version__', None)
    ROOT = Path('/content/drive/MyDrive/hsbi-toolkit')
    REPO = ROOT / 'hnsbi-toolkit'
    ROOT.mkdir(parents=True, exist_ok=True)
    if not (REPO / '.git').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/rafaellopesdesa/hnsbi-toolkit.git', str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[lhc,flows]'], check=True)
    installed_numpy_version = installed_version('numpy')
    if loaded_numpy_version not in (None, installed_numpy_version):
        print(
            f'NumPy changed from {loaded_numpy_version} to {installed_numpy_version}. '
            'Restarting the Colab runtime once to load a consistent binary stack. '
            'After it reconnects, run this setup cell again.',
            flush=True,
        )
        os.kill(os.getpid(), signal.SIGKILL)
else:
    REPO = Path.cwd()
    if not (REPO / 'pyproject.toml').exists():
        REPO = Path.cwd().parents[1]
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'examples' / 'lhc_analysis'))
print(REPO)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/hsbi-toolkit/hnsbi-toolkit


## Generate the configured samples

The shared generator supplies signal, background, and reference samples plus response, resolution, and signal-theory variations. Increase the event counts for publication runs.

In [2]:
from generate_distributions import generate
from hnsbi import Project

EXAMPLE = REPO / 'examples' / 'lhc_analysis'
generate(EXAMPLE / 'data', signal_events=120_000, background_events=300_000, reference_events=400_000)
project = Project.load(EXAMPLE / 'analysis.yaml')
print(project.config.features)


('x1', 'x2', 'x3', 'x4', 'x5')


## Train the reference and native ratio ensembles

Each ratio member has independent class normalization, deterministic train/validation/holdout splits, embedded preprocessing in ONNX, parity checks, and loss/overtraining/calibration/reweighting/normalization diagnostics.

In [3]:
reference_artifacts = project.train_reference()
reference = reference_artifacts.training.flow
ratio_artifacts = project.train_ratios(reference, normalization_events=40_000, seed=20260729)
for sample, training in ratio_artifacts.training.items():
    print(sample, training.manifest_path)
    print(training.members[0].metadata.get('diagnostics'))


signal /content/drive/MyDrive/hsbi-toolkit/hnsbi-toolkit/examples/lhc_analysis/artifacts/ratios/signal/ratio_ensemble.manifest.json
{'loss': {'epochs': 42, 'best_validation_loss': 0.6310197710990906, 'final_training_loss': 0.6297829747200012}, 'classification': {'holdout_auc': 0.6582496319446429}, 'overtraining': {'numerator_weighted_ks': 0.010916666666082486, 'denominator_weighted_ks': 0.006163461538743531}, 'calibration': {'brier_score': 0.22357933948321582, 'expected_calibration_error': 0.009599293979940587, 'training_brier_score': 0.22217564061463033, 'training_expected_calibration_error': 0.003852556867267609, 'holdout_log_ratio_weighted_rmse': 0.0734120473291253, 'training_log_ratio_weighted_rmse': 0.06155817576470694}, 'ratio_tails': {'training': {'minimum': 0.012112047404771512, 'maximum': 2.565159027854496, 'p001': 0.01584009036516411, 'p999': 2.240485207862585}, 'holdout': {'minimum': 0.012507112385678192, 'maximum': 2.5931754934356146, 'p001': 0.014878525225526405, 'p999': 2

## Systematics, Asimov closure, workspace, and fit

The Asimov weights use sample-wise $E_q[r_k]$ normalization. The workspace is JSON, while the first user interface remains YAML.

In [4]:
from hnsbi.inference import MinuitInference

systematic_training = project.train_systematics()
runtime_systematics = project.build_runtime_systematics(systematic_training)
asimov = project.build_configured_asimov(reference=reference, ratios=ratio_artifacts.evaluators, normalizer=ratio_artifacts.normalizer, systematics=runtime_systematics)
workspace = project.write_configured_workspace(asimov, reference_manifest=reference_artifacts.checkpoint_manifest, ratio_manifests={name: value.manifest_path for name, value in ratio_artifacts.training.items()})
likelihood = project.workspace_runtime(workspace.path)
fit = MinuitInference(likelihood).fit()
print('raw count / ESS:', asimov.raw_count, asimov.ess)
print(fit.point, fit.errors)


/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


XlaRuntimeError: INVALID_ARGUMENT: Unexpected PJRT_ExecuteOptions size: expected 88, got 80. The plugin is likely built with a later version than the framework. This plugin is built with PJRT API version 0.76.